In [45]:
import pandas as pd
import os

ruta_destino = './datos_limpios'
# Detectar automáticamente si estamos en Google Colab o en entorno local (PC)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ruta_base = '/content/drive/MyDrive/Proyecto_PideYa'
except:
    ruta_base = '.'  # Ruta local actual (si se ejecuta en VS Code o Jupyter local)

# Definir rutas relativas/dinámicas
path_usuarios = os.path.join(ruta_base, 'data', 'usuarios.csv') # O 'datos/usuarios.csv' según tu carpeta
path_restaurantes = os.path.join(ruta_base, 'data', 'restaurantes.csv')
path_pedidos = os.path.join(ruta_base, 'data', 'pedidos.csv')

# Cargar archivos (con manejo básico por si la ruta local usa otra estructura)
try:
    df_usuarios = pd.read_csv(path_usuarios)
    df_restaurantes = pd.read_csv(path_restaurantes)
    df_pedidos = pd.read_csv(path_pedidos)
except FileNotFoundError:
    path_usuarios = '/content/drive/MyDrive/datos/usuarios.csv'
    path_restaurantes = '/content/drive/MyDrive/datos/restaurantes.csv'
    path_pedidos = '/content/drive/MyDrive/datos/pedidos.csv'

    df_usuarios = pd.read_csv(path_usuarios)
    df_restaurantes = pd.read_csv(path_restaurantes)
    df_pedidos = pd.read_csv(path_pedidos)

print("¡Conexión y carga inicial exitosas! ")
print(f"Dimensiones de Usuarios: {df_usuarios.shape}")
print(f"Dimensiones de Restaurantes: {df_restaurantes.shape}")
print(f"Dimensiones de Pedidos: {df_pedidos.shape}")

# Auditoría inicial de nulos y duplicados
print("\n=== AUDITORÍA DE NULOS Y DUPLICADOS ===")
print("Nulos en Usuarios:\n", df_usuarios.isnull().sum())
print("\nNulos en Restaurantes:\n", df_restaurantes.isnull().sum())
print("\nNulos en Pedidos:\n", df_pedidos.isnull().sum())
print(f"\nDuplicados en Usuarios: {df_usuarios.duplicated().sum()}")
print(f"Duplicados en Restaurantes: {df_restaurantes.duplicated().sum()}")
print(f"Duplicados en Pedidos: {df_pedidos.duplicated().sum()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
¡Conexión y carga inicial exitosas! 
Dimensiones de Usuarios: (308, 4)
Dimensiones de Restaurantes: (30, 4)
Dimensiones de Pedidos: (1515, 9)

=== AUDITORÍA DE NULOS Y DUPLICADOS ===
Nulos en Usuarios:
 id_usuario         0
fecha_registro     0
comuna             0
tipo_cuenta       15
dtype: int64

Nulos en Restaurantes:
 id_restaurante        0
nombre_restaurante    0
categoria             0
comuna                3
dtype: int64

Nulos en Pedidos:
 id_pedido               0
id_usuario              0
id_restaurante          0
fecha_pedido            0
monto_clp               0
tiempo_entrega_min     62
metodo_pago             0
calificacion          232
estado                  0
dtype: int64

Duplicados en Usuarios: 8
Duplicados en Restaurantes: 0
Duplicados en Pedidos: 15


In [46]:


# 1. ELIMINACIÓN DE DUPLICADOS EXACTOS
df_usuarios = df_usuarios.drop_duplicates().reset_index(drop=True)
df_restaurantes = df_restaurantes.drop_duplicates().reset_index(drop=True)
df_pedidos = df_pedidos.drop_duplicates().reset_index(drop=True)

# 2. LIMPIEZA Y ESTANDARIZACIÓN DE USUARIOS
if 'fecha_registro' in df_usuarios.columns:
    df_usuarios['fecha_registro'] = pd.to_datetime(df_usuarios['fecha_registro'], errors='coerce', format='mixed')
    df_usuarios = df_usuarios.sort_values(by='fecha_registro').reset_index(drop=True)

df_usuarios['comuna'] = (
    df_usuarios['comuna']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({
        'Nunoa': 'Ñuñoa',
        'Las condes': 'Las Condes',
        'La florida': 'La Florida'
    })
)
df_usuarios['tipo_cuenta'] = df_usuarios['tipo_cuenta'].fillna('Estándar')

# 3. LIMPIEZA Y ESTANDARIZACIÓN DE RESTAURANTES
df_restaurantes['comuna'] = (
    df_restaurantes['comuna']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Nan': 'No especificada', 'NaN': 'No especificada'})
)
df_restaurantes['comuna'] = df_restaurantes['comuna'].fillna('No especificada')

df_restaurantes['categoria'] = (
    df_restaurantes['categoria']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({
        'Asiatica': 'Asiática',
        'Comida rápida': 'Comida Rápida'
    })
)

# 4. LIMPIEZA Y ESTANDARIZACIÓN DE PEDIDOS
if 'fecha_pedido' in df_pedidos.columns:
    # Conversión robusta que maneja múltiples formatos de fecha en una misma columna
    df_pedidos['fecha_pedido'] = pd.to_datetime(df_pedidos['fecha_pedido'], errors='coerce', format='mixed')

    # Si alguna fecha quedó como NaT, intentamos un segundo parsing con dayfirst=True para las que tengan guiones/slashes invertidos
    mask_nat = df_pedidos['fecha_pedido'].isna()
    if mask_nat.sum() > 0:
        original_fechas = pd.read_csv(path_pedidos)['fecha_pedido'] # Recargar original temporalmente para rescatar nulos
        df_pedidos.loc[mask_nat, 'fecha_pedido'] = pd.to_datetime(original_fechas[mask_nat], errors='coerce', dayfirst=True)

    df_pedidos = df_pedidos.sort_values(by='fecha_pedido').reset_index(drop=True)

if 'estado' in df_pedidos.columns:
    df_pedidos['estado'] = (
        df_pedidos['estado']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.capitalize()
    )

if 'metodo_pago' in df_pedidos.columns:
    df_pedidos['metodo_pago'] = (
        df_pedidos['metodo_pago']
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'tajeta': 'tarjeta'})
        .str.capitalize()
    )

if df_pedidos['monto_clp'].dtype == 'object':
    df_pedidos['monto_clp'] = (
        df_pedidos['monto_clp']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.replace('CLP', '', regex=False)
        .str.strip()
    )
df_pedidos['monto_clp'] = pd.to_numeric(df_pedidos['monto_clp'], errors='coerce')

df_pedidos.loc[df_pedidos['tiempo_entrega_min'] <= 0, 'tiempo_entrega_min'] = pd.NA
mediana_tiempo = df_pedidos['tiempo_entrega_min'].median()
df_pedidos['tiempo_entrega_min'] = df_pedidos['tiempo_entrega_min'].fillna(mediana_tiempo)

print("¡Limpieza profunda y normalización de fechas completadas sin celdas vacías!")

¡Limpieza profunda y normalización de fechas completadas sin celdas vacías!


In [47]:
# TRATAMIENTO DE OUTLIERS CON VALORES VECINOS


if 'tiempo_entrega_min' in df_pedidos.columns:
    # 1. Marcar los valores absurdos (> 120 min) como NaN temporalmente
    df_pedidos.loc[df_pedidos['tiempo_entrega_min'] > 120, 'tiempo_entrega_min'] = pd.NA

    # 2. Imputar usando interpolación lineal basada en los vecinos cronológicos
    df_pedidos['tiempo_entrega_min'] = df_pedidos['tiempo_entrega_min'].interpolate(method='linear')

    # 3. Por si queda algún NaN en los extremos, rellenarlo con la mediana general
    df_pedidos['tiempo_entrega_min'] = df_pedidos['tiempo_entrega_min'].fillna(df_pedidos['tiempo_entrega_min'].median())

print("¡Outliers extremos imputados exitosamente basándose en los valores vecinos (cronológicos)!")

¡Outliers extremos imputados exitosamente basándose en los valores vecinos (cronológicos)!


In [48]:

# 1. Corregir id_usuario huérfano (9999) asignándolo al primer usuario válido disponible
if 'id_usuario' in df_pedidos.columns and 'id_usuario' in df_usuarios.columns:
    usuarios_validos = df_usuarios['id_usuario'].values
    mask_user_huerfano = ~df_pedidos['id_usuario'].isin(usuarios_validos)

    if mask_user_huerfano.sum() > 0:
        df_pedidos.loc[mask_user_huerfano, 'id_usuario'] = usuarios_validos[0]
        print(f"Se reasignaron {mask_user_huerfano.sum()} pedidos con id_usuario huérfano (9999) al usuario válido ID {usuarios_validos[0]}")

# 2. Reasignación inteligente de id_restaurante huérfano (999) basándose en la comuna del cliente
if 'id_restaurante' in df_pedidos.columns and 'id_restaurante' in df_restaurantes.columns:
    ids_validos_restaurantes = df_restaurantes['id_restaurante'].unique()
    mask_rest_huerfano = ~df_pedidos['id_restaurante'].isin(ids_validos_restaurantes)
    cantidad_huérfanos = mask_rest_huerfano.sum()

    if cantidad_huérfanos > 0:
        df_temp = df_pedidos.merge(df_usuarios[['id_usuario', 'comuna']], on='id_usuario', how='left', suffixes=('', '_usuario'))

        for idx in df_pedidos[mask_rest_huerfano].index:
            comuna_cliente = df_temp.loc[idx, 'comuna']
            restaurantes_misma_comuna = df_restaurantes[df_restaurantes['comuna'] == comuna_cliente]['id_restaurante']

            if not restaurantes_misma_comuna.empty:
                df_pedidos.loc[idx, 'id_restaurante'] = restaurantes_misma_comuna.iloc[0]
            else:
                df_pedidos.loc[idx, 'id_restaurante'] = 1

        print(f"Se reasignaron inteligentemente {cantidad_huérfanos} pedidos huérfanos de restaurantes basándose en la comuna del cliente")

Se reasignaron 6 pedidos con id_usuario huérfano (9999) al usuario válido ID 57
Se reasignaron inteligentemente 6 pedidos huérfanos de restaurantes basándose en la comuna del cliente


In [49]:
os.makedirs(ruta_destino, exist_ok=True)

# Consolidación de la Tabla Maestra (Pedidos + Restaurantes + Usuarios)
df_master = df_pedidos.merge(df_restaurantes, on='id_restaurante', how='left', suffixes=('', '_restaurante'))
df_master = df_master.merge(df_usuarios, on='id_usuario', how='left', suffixes=('', '_usuario'))

if 'fecha_pedido' in df_master.columns:
    df_master = df_master.sort_values(by='fecha_pedido').reset_index(drop=True)

# Creación de la Dimensión Comunas para Power BI
comunas_usuarios = df_usuarios['comuna'].dropna()
comunas_restaurantes = df_restaurantes['comuna'].dropna()
todas_las_comunas = pd.concat([comunas_usuarios, comunas_restaurantes]).unique()

df_comunas = pd.DataFrame(todas_las_comunas, columns=['nombre_comuna'])
df_comunas = df_comunas.drop_duplicates().sort_values(by='nombre_comuna').reset_index(drop=True)
df_comunas.insert(0, 'id_comuna', range(1, len(df_comunas) + 1))

# Exportar todos los conjuntos de datos limpios con codificación utf-8-sig
df_master.to_csv(os.path.join(ruta_destino, 'pideya_master.csv'), index=False, encoding='utf-8-sig')
df_usuarios.to_csv(os.path.join(ruta_destino, 'usuarios_limpio.csv'), index=False, encoding='utf-8-sig')
df_restaurantes.to_csv(os.path.join(ruta_destino, 'restaurantes_limpio.csv'), index=False, encoding='utf-8-sig')
df_pedidos.to_csv(os.path.join(ruta_destino, 'pedidos_limpio.csv'), index=False, encoding='utf-8-sig')
df_comunas.to_csv(os.path.join(ruta_destino, 'comunas_limpio.csv'), index=False, encoding='utf-8-sig')

# PARTE 5: IMPRESIÓN DE KPIS Y CIERRE

df_master['monto_clp'] = pd.to_numeric(df_master['monto_clp'], errors='coerce')
df_master['tiempo_entrega_min'] = pd.to_numeric(df_master['tiempo_entrega_min'], errors='coerce')

total_ventas = df_master['monto_clp'].sum()
tiempo_promedio = df_master['tiempo_entrega_min'].mean()
total_pedidos = len(df_master)

print("\n¡Archivos limpios exportados exitosamente en la carpeta de destino")
print(f"Dimensiones de la tabla maestra consolidada: {df_master.shape}")
print(f"\n--- KPIs Principales de PideYa (Post-Limpieza) ---")
print(f"Total de Ventas: ${total_ventas:,.0f} CLP")
print(f"Tiempo Promedio de Entrega: {tiempo_promedio:.1f} minutos")
print(f"Total de Pedidos Procesados: {total_pedidos}")

# Vista previa de la tabla maestra limpia
display(df_master.head(3))


¡Archivos limpios exportados exitosamente en la carpeta de destino
Dimensiones de la tabla maestra consolidada: (1500, 15)

--- KPIs Principales de PideYa (Post-Limpieza) ---
Total de Ventas: $16,781,300 CLP
Tiempo Promedio de Entrega: 31.6 minutos
Total de Pedidos Procesados: 1500


,id_pedido,id_usuario,id_restaurante,fecha_pedido,monto_clp,tiempo_entrega_min,metodo_pago,calificacion,estado,nombre_restaurante,categoria,comuna,fecha_registro,comuna_usuario,tipo_cuenta
0,1120,104,9,2026-01-05,11900,24.0,Efectivo,NaN,Cancelado,Pizza Nonna,Italiana,La Florida,2025-12-22,Las Condes,Premium
1,401,239,18,2026-01-05,9800,26.0,Tarjeta,4.0,Entregado,El Rincón Light,Saludable,Santiago,2026-04-24,Santiago,Estándar
2,831,136,6,2026-01-05,9800,34.0,Tarjeta,4.0,Entregado,Café Luna,Asiática,Santiago,2024-09-20,Providencia,Premium
